# 📝 W2: 계약서/공문 자동화
**hexa-5 — 건설업 AX 마스터클래스 — Week 2**

---
**학습 목표**
1. 건설 현장 기본 정보만 입력하면 표준 계약서와 공문서를 자동 생성한다
2. 다수 프로젝트의 법률 문서를 배치로 한번에 만든다
3. 결과를 마크다운·텍스트 파일로 저장하여 바로 활용한다

> ⏱️ 예상 소요시간: 60분 | 🎯 결과물: 표준 계약서 초안 N건 (ZIP 다운로드)

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
!pip install -q google-generativeai pandas
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
_n=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _n: fm.fontManager.addfont(_n[0]); plt.rcParams['font.family']=fm.FontProperties(fname=_n[0]).get_name()
plt.rcParams['axes.unicode_minus']=False
try:
    from google.colab import userdata; API_KEY=userdata.get('GEMINI_API_KEY')
except: API_KEY=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=API_KEY)
# === 2026년 3월 기준 최신 모델 선택 ===
# ✅ GA(안정): gemini-2.5-flash  ← 기본값 (권장)
# 🔵 Preview : gemini-3.1-flash-lite-preview  (최신, 2026.3 출시)
# 🔵 Preview : gemini-3.1-pro-preview  (최강, 복잡한 분석용)
# ⚠️  구버전 gemini-2.0-flash 는 2026년 6월 1일 종료 예정
MODEL_NAME = 'gemini-2.5-flash'  # 원하는 모델로 변경하세요
model=genai.GenerativeModel('gemini-2.5-flash'); print('✅ 준비 완료')

## Step 1: 계약서 자동 생성 함수

In [ ]:
def generate_contract(project_name, client, contractor, amount, start_date, end_date, special_terms=""):
    """건설 공사 표준 계약서 생성"""
    
    contract_template = f"""
# 건설 공사 표준 계약서

## 계약 일반사항
- **공사명**: {project_name}
- **발주처**: {client}
- **시공사**: {contractor}
- **계약금액**: {amount:,}원 (부가세 포함)
- **착공일**: {start_date}
- **준공일**: {end_date}
- **공사기간**: {(end_date - start_date).days}일

## 제1조 공사의 내용
1. 본 계약의 공사 내용은 첨부된 시방서 및 도면에 따른다.
2. 시공사는 설계도면에 따라 공사를 시공하고 품질 관리를 책임진다.

## 제2조 계약금액과 지급 조건
1. 총 계약금액은 {amount:,}원이다.
2. 계약금 지급은 다음과 같이 분할 지급한다.
   - 착공금: 총액의 30%
   - 중기금: 총액의 40%
   - 준공금: 총액의 30%

## 제3조 공사 기간
1. 공사 기간은 {start_date}부터 {end_date}까지이다.
2. 천재지변이나 발주처의 귀책사유로 지연 시 기간 연장이 가능하다.

## 제4조 품질 보증
1. 시공사는 준공 후 하자보수기간 동안 책임을 진다.
2. 하자보수기간은 준공일로부터 1년으로 한다.

## 제5조 특약 조항
{special_terms if special_terms else '특별한 특약 조항 없음'}

## 서명

발주처: {client} (인)

시공사: {contractor} (인)

계약일: {start_date}
"""
    
    return contract_template

## Step 2: 공문서 자동 생성 함수

In [ ]:
def generate_official_letter(recipient, sender, subject, content, date, reference_no=""):
    """건설회사 공문서 생성"""
    
    letter_template = f"""
문서번호: {reference_no if reference_no else 'AZ-' + date.strftime('%Y%m%d') + '-001'}
수신: {recipient}
발신: {sender}
제목: {subject}
발신일: {date.strftime('%Y년 %m월 %d일')}

─────────────────────────────────────

{content}

─────────────────────────────────────

위와 같이 통보하오니 알아두시기 바랍니다.

건설현장 안전과 품질 확보에 협조해 주시기 바랍니다.

감사합니다.


〔sender〕
건설회사 대표이사 (서명)
"""
    
    return letter_template

## Step 3: Gemini AI를 활용한 계약서 개선

In [ ]:
def enhance_contract_with_ai(contract_text, project_type="일반건축"):
    """Gemini AI로 계약서 특화 및 개선"""
    
    prompt = f"""
다음 건설 계약서 초안을 {project_type} 프로젝트에 맞게 전문적으로 개선해주세요.

요구사항:
1. {project_type}의 특수성을 반영한 조항 추가
2. 법적 효력이 있는 문언으로 수정
3. 실무에서 사용하는 전문 용어로 변경
4. 위험 관리 및 분쟁 예방 조항 강화

원본 계약서:
{contract_text}

개선된 계약서를 마크다운 형식으로 작성해주세요.
"""
    
    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        print(f"AI 개선 중 오류 발생: {e}")
        return contract_text

## Step 4: 샘플 데이터 생성

In [ ]:
import pandas as pd
from datetime import datetime, timedelta

# 샘플 프로젝트 데이터
projects = [
    {
        'project_name': '송파구 오피스빌딩 신축공사',
        'client': '㈜송파개발공사',
        'contractor': '㈜한빛건설',
        'amount': 2500000000,
        'start_date': datetime(2024, 3, 1),
        'end_date': datetime(2024, 8, 31),
        'project_type': '상업용 건축',
        'special_terms': '내부 인테리어 공사는 별도 계약으로 진행함'
    },
    {
        'project_name': '강남구 아파트 리모델링',
        'client': '강남주택관리조합',
        'contractor': '㈜미래건설',
        'amount': 1800000000,
        'start_date': datetime(2024, 4, 15),
        'end_date': datetime(2024, 10, 15),
        'project_type': '주거용 리모델링',
        'special_terms': '입주민 이사 일정에 따른 공사 단계별 시행'
    },
    {
        'project_name': '판교 테크밸리 공장 증축',
        'client': '㈜판교전자',
        'contractor': '㈜기술건설',
        'amount': 3200000000,
        'start_date': datetime(2024, 5, 1),
        'end_date': datetime(2024, 12, 31),
        'project_type': '산업시설',
        'special_terms': '생산 중단 최소화를 위한 야간 공사 우선 시행'
    }
]

df_projects = pd.DataFrame(projects)
print('📋 프로젝트 목록:')
display(df_projects[['project_name', 'client', 'contractor', 'amount']])

## Step 5: 계약서 생성 및 AI 개선

In [ ]:
generated_contracts = []

for i, project in enumerate(projects):
    print(f'\n🔹 [{i+1}/{len(projects)}] {project["project_name"]} 계약서 생성 중...')
    
    # 기본 계약서 생성
    contract = generate_contract(
        project['project_name'],
        project['client'], 
        project['contractor'],
        project['amount'],
        project['start_date'],
        project['end_date'],
        project['special_terms']
    )
    
    # AI 개선
    enhanced_contract = enhance_contract_with_ai(contract, project['project_type'])
    
    generated_contracts.append({
        'project_name': project['project_name'],
        'contract': enhanced_contract
    })
    
    print(f'✅ {project["project_name"]} 계약서 생성 완료')

print(f'\n🎉 총 {len(generated_contracts)}건의 계약서 생성 완료!')

## Step 6: 공문서 생성 예시

In [ ]:
# 공문서 샘플 생성
sample_letters = [
    {
        'recipient': '서울시 송파구청장',
        'sender': '㈜한빛건설',
        'subject': '건축물 사용승인 신고서',
        'content': '''귀청 관내 소재 부지에 진행 중인 '송파구 오피스빌딩 신축공사'가 2024년 8월 31일자로 준공되었음을 알려드립니다.

관련 법규에 따라 건축법 제29조 및 동법 시행령 제22조에 의거 건축물 사용승인을 신고하오니 검토 후 조치하여 주시기 바랍니다.

첨부서류:
1. 준공검사조서 1부
2. 설계도면 일체
3. 시공자 겸임 확인서 1부''',
        'date': datetime(2024, 9, 5),
        'reference_no': 'AZ-20240905-001'
    },
    {
        'recipient': '강남구청 안전관리과장',
        'sender': '㈜미래건설',
        'subject': '건설 현장 안전점검 결과 보고',
        'content': '''2024년 9월 3일 실시한 '강남구 아파트 리모델링' 현장 안전점검 결과를 보고합니다.

점검 결과:
- 안전관리계획서 적정: 양호
- 안전시설물 설치: 완료
- 근로자 안전보건교육 이수: 100%
- 개선 필요 사항: 2건 (조치 완료)

향후도 안전관리에 만전을 기하겠습니다.''',
        'date': datetime(2024, 9, 4),
        'reference_no': 'AZ-20240904-002'
    }
]

for i, letter in enumerate(sample_letters):
    official_letter = generate_official_letter(
        letter['recipient'],
        letter['sender'],
        letter['subject'],
        letter['content'],
        letter['date'],
        letter['reference_no']
    )
    
    print(f'\n📄 [{i+1}] {letter["subject"]}')
    print('-' * 50)
    print(official_letter[:400] + '...' if len(official_letter) > 400 else official_letter)

## Step 7: 파일 저장 및 다운로드

In [ ]:
import os
import zipfile
from google.colab import files

# 저장할 디렉토리 생성
!mkdir -p contracts_documents

# 계약서 파일 저장
for contract_data in generated_contracts:
    filename = f"contracts_documents/계약서_{contract_data['project_name'].replace('/', '_')}.md"
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(contract_data['contract'])

# 공문서 파일 저장
for i, letter in enumerate(sample_letters):
    filename = f"contracts_documents/공문서_{letter['subject'].replace('/', '_')}.md"
    official_letter = generate_official_letter(
        letter['recipient'],
        letter['sender'],
        letter['subject'],
        letter['content'],
        letter['date'],
        letter['reference_no']
    )
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(official_letter)

# ZIP 파일 생성
with zipfile.ZipFile('건설_계약서_공문서_모음.zip', 'w') as zipf:
    for foldername, subfolders, filenames in os.walk('contracts_documents'):
        for filename in filenames:
            filepath = os.path.join(foldername, filename)
            zipf.write(filepath, filename)

print('✅ 모든 문서가 성공적으로 저장되었습니다.')
print('📁 생성된 파일 목록:')
for filename in os.listdir('contracts_documents'):
    print(f'  - {filename}')

# 다운로드
files.download('건설_계약서_공문서_모음.zip')
print('\n🎉 ZIP 파일 다운로드 완료!')

## Step 8: 계약서 생성 통계

In [ ]:
# 생성된 문서 통계
stats = {
    '총 계약서': len(generated_contracts),
    '총 공문서': len(sample_letters),
    '총 계약금액': sum(p['amount'] for p in projects),
    '평균 공사기간': sum((p['end_date'] - p['start_date']).days for p in projects) // len(projects)
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# 프로젝트별 계약금액
project_names = [p['project_name'][:20] + '...' if len(p['project_name']) > 20 else p['project_name'] for p in projects]
amounts = [p['amount']/100000000 for p in projects]  # 억 단위 변환

bars1 = ax1.bar(project_names, amounts, color='skyblue', alpha=0.8)
ax1.set_title('프로젝트별 계약금액', fontsize=14, fontweight='bold')
ax1.set_ylabel('계약금액 (억원)', fontsize=12)
ax1.tick_params(axis='x', rotation=45)

# 막대에 수치 표시
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height):,}억', ha='center', va='bottom')

# 문서 생성 통계
categories = ['계약서', '공문서']
values = [stats['총 계약서'], stats['총 공문서']]
colors = ['lightcoral', 'lightgreen']

bars2 = ax2.bar(categories, values, color=colors, alpha=0.8)
ax2.set_title('생성된 문서 통계', fontsize=14, fontweight='bold')
ax2.set_ylabel('문서 수', fontsize=12)

# 막대에 수치 표시
for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}건', ha='center', va='bottom')

plt.tight_layout()
plt.show()

# 통계 요약
print('\n📊 생성 통계 요약:')
print(f'• 총 생성 문서: {stats["총 계약서"] + stats["총 공문서"]}건')
print(f'• 총 계약금액: {stats["총 계약금액"]:,}원 ({stats["총 계약금액"]//100000000}억원)')
print(f'• 평균 공사기간: {stats["평균 공사기간"]}일')